# Name That LLM: Starter Baseline (TF-IDF + Logistic Regression)


In [1]:
import pathlib

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline

CANDIDATES = [
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('/kaggle/input/name-that-llm'),
]
DATA_DIR = next(p for p in CANDIDATES if (p / 'train.csv').exists())

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train_df.shape} | Unique models: {train_df["model"].nunique()} | Prompts: {train_df["prompt_id"].nunique()}')
print(f'Test shape: {test_df.shape}')
train_df.head(3)

Train shape: (2699, 4) | Unique models: 18 | Prompts: 150
Test shape: (3, 2)


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash


In [2]:
# Build Word + Character N-Gram TF-IDF + Logistic Regression Pipeline
model_pipeline = Pipeline(
    [
        (
            'features',
            FeatureUnion(
                [
                    (
                        'word_tfidf',
                        TfidfVectorizer(
                            ngram_range=(1, 2),
                            max_features=25000,
                            sublinear_tf=True,
                        ),
                    ),
                    (
                        'char_tfidf',
                        TfidfVectorizer(
                            analyzer='char',
                            ngram_range=(3, 5),
                            max_features=35000,
                            sublinear_tf=True,
                        ),
                    ),
                ]
            ),
        ),
        ('clf', LogisticRegression(C=2.0, max_iter=1000)),
    ]
)

# 5-fold CV that keeps each prompt in a single fold (test prompts are unseen), scored like the leaderboard
cv_scores = cross_val_score(
    model_pipeline,
    train_df['text'].fillna(''),
    train_df['model'],
    groups=train_df['prompt_id'],
    cv=GroupKFold(n_splits=5),
    scoring='balanced_accuracy',
    n_jobs=-1,
)
print(f'5-Fold Grouped CV Balanced Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Fit on full training set
model_pipeline.fit(train_df['text'].fillna(''), train_df['model'])

5-Fold Grouped CV Balanced Accuracy: 0.3145 +/- 0.0153


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=25000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=35000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 sublinear_tf=True))])),
                ('clf', LogisticRegression(C=2.0, max_iter=1000))])

In [3]:
# Batch inference on test.csv (1 example row interactively; the full hidden test set during the Leaderboard Rerun)
test_preds = model_pipeline.predict(test_df['text'].fillna(''))

submission = pd.DataFrame(
    {
        'id': test_df['id'],
        'model': test_preds,
    }
)

submission.to_parquet('submission.parquet', index=False)
print(f'Saved submission.parquet with {len(submission)} rows.')
submission.head(10)

Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,gemini-3.8-flash
1,005f5a9a6139,gpt-5.6-terra
2,0062a53d5487,gemini-3.7-flash
